In [1]:
import os
import zipfile
import csv
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.transforms import v2
import torchvision.transforms.functional as TF
from sklearn.model_selection import train_test_split  # added import



In [2]:
data_root = "/content/denoising_data"
os.makedirs(data_root, exist_ok=True)
zip_paths = [
    "/kaggle/input/denoising-dirty-documents/train.zip",
    "/kaggle/input/denoising-dirty-documents/test.zip",
    "/kaggle/input/denoising-dirty-documents/train_cleaned.zip",
]
for zp in zip_paths:
    with zipfile.ZipFile(zp, "r") as zf:
        zf.extractall(data_root)



In [3]:
train_dir = os.path.join(data_root, "train")
train_cleaned_dir = os.path.join(data_root, "train_cleaned")
test_dir = os.path.join(data_root, "test")




In [4]:
class PadToSize:
    def __init__(self, target_size):
        self.target_size = target_size  # (height, width)

    def __call__(self, img):
        _, height, width = img.shape
        target_height, target_width = self.target_size
        pad_top = (target_height - height) // 2
        pad_bottom = target_height - height - pad_top
        pad_left = (target_width - width) // 2
        pad_right = target_width - width - pad_left
        return TF.pad(img, [pad_left, pad_top, pad_right, pad_bottom], fill=0)


class Grayscale:
    def __call__(self, img):
        pil_img = TF.to_pil_image(img) if isinstance(img, torch.Tensor) else img
        grayscale_img = pil_img.convert("L")
        return TF.to_tensor(grayscale_img)


train_transforms = v2.Compose(
    [
        v2.ToImage(),
        PadToSize((420, 540)),
        v2.RandomApply([v2.GaussianBlur(kernel_size=3)], p=0.4),
        v2.RandomApply([v2.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        Grayscale(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

val_transforms = v2.Compose(
    [
        v2.ToImage(),
        PadToSize((420, 540)),
        Grayscale(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)

test_transforms = v2.Compose(
    [
        v2.ToImage(),
        PadToSize((420, 540)),
        Grayscale(),
        v2.ToDtype(torch.float32, scale=True),
    ]
)




In [5]:
class ImageDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.transform = transform
        self.image_files = sorted(
            [
                os.path.join(data_dir, f)
                for f in os.listdir(data_dir)
                if f.endswith(".png")
            ]
        )

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image


class PairedImageDataset(Dataset):
    def __init__(self, noisy_files, clean_files, transform=None):
        self.noisy_files = noisy_files
        self.clean_files = clean_files
        self.transform = transform

    def __len__(self):
        return len(self.noisy_files)

    def __getitem__(self, idx):
        noisy = Image.open(self.noisy_files[idx]).convert("RGB")
        clean = Image.open(self.clean_files[idx]).convert("RGB")
        if self.transform:
            noisy = self.transform(noisy)
            clean = self.transform(clean)
        return noisy, clean


all_noisy = sorted(
    [os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith(".png")]
)
all_clean = sorted(
    [
        os.path.join(train_cleaned_dir, f)
        for f in os.listdir(train_cleaned_dir)
        if f.endswith(".png")
    ]
)

train_noisy, val_noisy, train_clean, val_clean = train_test_split(
    all_noisy, all_clean, test_size=2 / 9, random_state=42
)

train_dataset = PairedImageDataset(train_noisy, train_clean, train_transforms)
val_dataset = PairedImageDataset(val_noisy, val_clean, val_transforms)
test_dataset = ImageDataset(test_dir, test_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)



In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


In [7]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, bias=False),
            nn.Conv2d(1, 8, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(8, 8, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(8, 16, kernel_size=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.MaxPool2d(2),
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(16, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.MaxPool2d(2),
        )
        self.enc4 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(32, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.MaxPool2d(2),
        )
        self.enc5 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.MaxPool2d(2),
        )
        self.dec5 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                64,
                64,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=64,
                bias=False,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                32,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=32,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                16,
                16,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=16,
                bias=False,
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(16, 8, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                8,
                8,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=8,
                bias=False,
            ),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(8, 1, kernel_size=1, bias=False),
            nn.ConvTranspose2d(
                1,
                1,
                kernel_size=3,
                stride=2,
                padding=1,
                output_padding=1,
                groups=1,
                bias=False,
            ),
            nn.Sigmoid(),
        )
        self.conv4 = nn.Conv2d(64, 32, kernel_size=1, bias=False)
        self.conv5 = nn.Conv2d(128, 64, kernel_size=1, bias=False)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        d5 = self.dec5(e5)
        d5 = F.interpolate(d5, size=(26, 33), mode="bilinear", align_corners=False)
        d5 = torch.cat([d5, e4], dim=1)
        d5 = self.conv5(d5)

        d4 = self.dec4(e4)
        d4 = F.interpolate(d4, size=(52, 67), mode="bilinear", align_corners=False)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = self.conv4(d4)

        d3 = self.dec3(d4)
        d3 = F.interpolate(d3, size=(105, 135), mode="bilinear", align_corners=False)

        d2 = self.dec2(d3)
        d2 = F.interpolate(d2, size=(210, 270), mode="bilinear", align_corners=False)

        d1 = self.dec1(d2)
        d1 = F.interpolate(d1, size=(420, 540), mode="bilinear", align_corners=False)

        return torch.clamp(d1, min=0.001, max=0.999)


model = DenoisingAutoencoder().to(device)




In [8]:
class RMSELoss(nn.Module):
    def forward(self, pred, target):
        return torch.sqrt(F.mse_loss(pred, target))


class HybridLoss(nn.Module):
    def __init__(self, lambda_rmse=0.8, lambda_l1=0.2):
        super().__init__()
        self.lambda_rmse = lambda_rmse
        self.lambda_l1 = lambda_l1
        self.rmse = RMSELoss()
        self.l1 = nn.L1Loss()

    def forward(self, pred, target):
        return self.lambda_rmse * self.rmse(pred, target) + self.lambda_l1 * self.l1(
            pred, target
        )


criterion = HybridLoss()
# Reduced learning rate for more stable convergence
optimizer = optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)



In [9]:
# Increased epochs and early‑stopping patience to allow better learning
epochs = 12  # was 5
patience = 4  # was 2
best_val = float("inf")
best_state = None
early_stop = 0

for epoch in range(epochs):
    model.train()
    train_losses = []
    for noisy, clean in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad()
        out = model(noisy)
        loss = criterion(out, clean)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    train_loss = np.mean(train_losses)

    model.eval()
    val_losses = []
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy, clean = noisy.to(device), clean.to(device)
            out = model(noisy)
            loss = criterion(out, clean)
            val_losses.append(loss.item())
    val_loss = np.mean(val_losses)

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.5f} - Val loss: {val_loss:.5f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        best_state = model.state_dict()
        early_stop = 0
    else:
        early_stop += 1
        if early_stop >= patience:
            print("Early stopping")
            break

if best_state is not None:
    torch.save(best_state, "best_model.pth")
    print("Best model saved.")
else:
    print("No model saved.")



Epoch 1/12 - Train loss: 0.44182 - Val loss: 0.47396
Epoch 2/12 - Train loss: 0.43161 - Val loss: 0.47299
Epoch 3/12 - Train loss: 0.42448 - Val loss: 0.47024
Epoch 4/12 - Train loss: 0.41391 - Val loss: 0.46527
Epoch 5/12 - Train loss: 0.38180 - Val loss: 0.45797
Epoch 6/12 - Train loss: 0.34051 - Val loss: 0.44919
Epoch 7/12 - Train loss: 0.31389 - Val loss: 0.43641
Epoch 8/12 - Train loss: 0.27444 - Val loss: 0.42531
Epoch 9/12 - Train loss: 0.25670 - Val loss: 0.39361
Epoch 10/12 - Train loss: 0.24157 - Val loss: 0.36399
Epoch 11/12 - Train loss: 0.22811 - Val loss: 0.35664
Epoch 12/12 - Train loss: 0.22357 - Val loss: 0.26570
Best model saved.


In [10]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()
all_outputs = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        preds = model(batch)  # (B,1,420,540)
        all_outputs.append(preds.cpu())
all_outputs = torch.cat(all_outputs, dim=0)  # (N,1,420,540)



In [11]:
target_size = (420, 540)


def compute_padding(orig_h, orig_w, tgt_h, tgt_w):
    pad_top = (tgt_h - orig_h) // 2 if tgt_h > orig_h else 0
    pad_left = (tgt_w - orig_w) // 2 if tgt_w > orig_w else 0
    return pad_top, pad_left


submission_rows = []
for idx, file_path in enumerate(test_dataset.image_files):
    img_id = os.path.splitext(os.path.basename(file_path))[0]
    pil = Image.open(file_path)
    orig_w, orig_h = pil.size

    pred = all_outputs[idx, 0]  # shape (420, 540)

    pad_top, pad_left = compute_padding(orig_h, orig_w, *target_size)
    cropped = pred[
        pad_top : pad_top + orig_h, pad_left : pad_left + orig_w
    ]  # (orig_h, orig_w)
    pred_np = cropped.numpy().astype(float)

    for r in range(orig_h):
        for c in range(orig_w):
            pixel_id = f"{img_id}_{r+1}_{c+1}"
            pixel_val = float(pred_np[r, c])
            submission_rows.append((pixel_id, pixel_val))

submission_path = "submission.csv"
with open(submission_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])
    writer.writerows(submission_rows)

print(f"Submission saved to {submission_path} with {len(submission_rows)} rows.")

Submission saved to submission.csv with 5789880 rows.
